# Task :  Build a multi‑agent system in CrewAI.
#### in Phase 3 - Module 7: Multi-Agent Systems
-------
that automates lead qualification from client outreach → research → scoring → summarisation/reporting.

## 1. Import Libraries

In [1]:
import os, warnings, asyncio, requests
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool
from crewai.tools import tool
from crewai.memory.unified_memory import Memory
import crewai.llms.cache as _crewai_cache
import nest_asyncio

nest_asyncio.apply()

_crewai_cache.mark_cache_breakpoint = lambda msg: msg
warnings.filterwarnings('ignore')

print('Libraries imported OK')

Libraries imported OK


## 2. Load Environment Variables

In [2]:
_p = Path.cwd()
for _ in range(6):
    if (_p / '.env').exists():
        load_dotenv(dotenv_path=_p / '.env', override=True)
        break
    _p = _p.parent
else:
    load_dotenv(override=True)

os.environ['CREWAI_STORAGE_DIR'] = str(Path.cwd() / '.crewai_storage')


## 3. Configure LLM

In [3]:
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
USE_SERPER = os.getenv('USE_SERPER', 'false').lower() == 'true'
SERPER_API_KEY = os.getenv('SERPER_API_KEY', '') if USE_SERPER else ''

if not GROQ_API_KEY:
    raise ValueError('No LLM API key found. Add GROQ_API_KEY to .env')

if GROQ_API_KEY:
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY

if SERPER_API_KEY:
    os.environ['SERPER_API_KEY'] = SERPER_API_KEY

if GROQ_API_KEY:
    selected_model = os.getenv('GROQ_MODEL', 'groq/qwen/qwen3.8-27b')
    llm = LLM(
        model=selected_model,
        api_key=GROQ_API_KEY,
        temperature=0.2,
        max_tokens=700
    )

print(f'LLM: {selected_model}')
print('Serper web search:', 'enabled' if SERPER_API_KEY else 'disabled')

# Local ONNX embeddings keep CrewAI memory enabled without Gemini/OpenAI embedding quota.
embedder_config = {
    "provider": "onnx",
    "config": {}
}

def create_agent_memory():
    return Memory(embedder=embedder_config, llm=llm)


LLM: groq/qwen/qwen3.8-27b
Serper web search: enabled


## 4. Define Tools

### 4.1 Internal Lead Database Tool

In [4]:
lead_df = pd.DataFrame([
    {'company': 'MedTech Solutions',  'industry': 'Healthcare', 'country': 'United States', 'employees': 250,  'annual_budget': '$200,000',   'notes': 'Interested in AI patient triage and healthcare automation.'},
    {'company': 'HealthPulse AI',     'industry': 'Healthcare', 'country': 'United States', 'employees': 120,  'annual_budget': '$95,000',    'notes': 'Looking for diagnostic assistant software.'},
    {'company': 'BioCare Innovation', 'industry': 'Healthcare', 'country': 'United States', 'employees': 500,  'annual_budget': '$500,000',   'notes': 'Major medical equipment supplier expanding into AI tools.'},
    {'company': 'CityRetail Group',   'industry': 'Retail',     'country': 'United Kingdom','employees': 800,  'annual_budget': '$1,200,000', 'notes': 'Looking for supply-chain AI solutions.'},
    {'company': 'FinEdge Capital',    'industry': 'Finance',    'country': 'United States', 'employees': 60,   'annual_budget': '$150,000',   'notes': 'Interested in fraud detection AI.'},
])

@tool('Lead Database Lookup')
def lead_database_lookup(industry: str, country: str) -> str:
    """Find potential leads by industry and country."""

    matches = lead_df[
        lead_df['industry'].str.contains(industry, case=False, na=False)
        & lead_df['country'].str.contains(country, case=False, na=False)
    ]

    if matches.empty:
        return 'No matching leads found.'

    return matches.to_string(index=False)


### 4.2 Web Search Tool (SerperDevTool)

In [5]:
if SERPER_API_KEY:
    @tool('Concise Serper Web Search')
    def concise_serper_search(search_query: str) -> str:
        """Search the web with Serper and return only concise top results."""
        response = requests.post(
            'https://google.serper.dev/search',
            headers={'X-API-KEY': SERPER_API_KEY, 'Content-Type': 'application/json'},
            json={'q': search_query, 'num': 3},
            timeout=20
        )
        response.raise_for_status()
        data = response.json()
        results = []
        for item in data.get('organic', [])[:3]:
            title = item.get('title', 'Untitled')
            link = item.get('link', 'No link')
            snippet = item.get('snippet', 'No snippet')[:220]
            results.append(f'- {title}: {snippet} Source: {link}')
        return '\n'.join(results) if results else 'No web results found.'

    research_tools = [lead_database_lookup, concise_serper_search]
    print('Tools ready: Lead DB Lookup + Concise Serper Web Search')
else:
    research_tools = [lead_database_lookup]
    print('Tools ready: Lead DB Lookup (Serper key not provided)')

Tools ready: Lead DB Lookup + Concise Serper Web Search


## 5. Define Agents

In [6]:
researcher_agent = Agent(
    role="Researcher Agent",
    goal=(
        'Find potential leads in the {industry} sector in {country}, '
        'research the identified companies, and prepare short personalized '
        'outreach messages.'
    ),
    backstory=(
        "You are a B2B sales researcher who researches potential clients "
        "and collects information to support lead qualification."
    ),
    tools=research_tools,
    memory=create_agent_memory(),
    verbose=True,
    max_iter=4,
    llm=llm
)


scoring_agent = Agent(
    role="Scoring Agent",
    goal=(
        'Evaluate the research findings for each identified lead and assign '
        "a lead qualification score from 0 to 100."
    ),
    backstory=(
        "You are a sales qualification analyst who evaluates leads based "
        "on budget, business need, company fit, and buying timeline."
    ),
    tools=[],
    memory=create_agent_memory(),
    verbose=True,
    max_iter=2,
    llm=llm
)


report_writer_agent = Agent(
    role="Report Writer Agent",
    goal=(
        'Write a concise Lead Qualification Report for all identified leads, '
        "combining the outreach messages, research findings, lead scores, "
        "and recommended next steps."
    ),
    backstory=(
        "You are a professional sales report writer who turns research "
        "and scoring results into clear qualification reports."
    ),
    tools=[],
    memory=create_agent_memory(),
    verbose=True,
    max_iter=2,
    llm=llm
)

## 6. Define Tasks & Workflow

In [7]:
research_task = Task(
    description=(
        'Find potential leads in the {industry} sector in {country}. '
        'Use the Lead Database Lookup tool to identify matching companies. '
        'Research each identified lead using at most one concise web search per company. '
        'Collect available information such as company size, annual budget, '
        'industry fit, and business needs. Prepare a short personalized '
        'outreach message for each lead. Keep findings concise and do not invent missing information.'
    ),
    expected_output=(
        'A structured list of potential leads with company name, industry, '
        'country, employee count, annual budget, business needs, research '
        'findings, and a short outreach message for each lead.'
    ),
    agent=researcher_agent
)

scoring_task = Task(
    description=(
        'Review the potential leads found by the Researcher Agent. '
        'Score each lead from 0 to 100 using: Budget (0-30), '
        'Need (0-30), Company Fit (0-20), and Timeline (0-20). '
        'Provide the total score, justification, and priority for each lead. '
        'Do not assume missing information.'
    ),
    expected_output=(
        'A qualification score for each lead with sub-scores, total score, '
        'justification, and priority: High (>=70), Medium (40-69), Low (<40).'
    ),
    agent=scoring_agent,
    context=[research_task]
)

report_task = Task(
    description=(
        'Create a final Lead Qualification Report using the research '
        'and scoring results. Include the identified leads, outreach messages, '
        'qualification scores, priorities, and recommended next steps. '
        'Do not add information that was not provided by previous tasks.'
    ),
    expected_output=(
        'A Markdown Lead Qualification Report containing all identified leads, '
        'research findings, outreach messages, scores, priorities, '
        'and recommended next steps.'
    ),
    agent=report_writer_agent,
    context=[research_task, scoring_task]
)

print('Tasks defined OK')


Tasks defined OK


## 7. Create Crew

In [8]:
# Create the Crew
crew = Crew(
    agents=[
        researcher_agent,
        scoring_agent,
        report_writer_agent
    ],
    tasks=[
        research_task,
        scoring_task,
        report_task
    ],
    process=Process.sequential,
    memory=True,
    embedder=embedder_config,
    verbose=True,
    max_rpm=1
)

print("Crew created OK")

Crew created OK


## 8. Run – Find Healthcare Leads in the US

In [ ]:
# Example: "Find potential leads in the healthcare sector in the US."
inputs = {
    'industry': 'Healthcare',
    'country': 'United States'
}

retry_delays = [70, 90]
result = None

for attempt in range(len(retry_delays) + 1):
    try:
        result = await crew.kickoff_async(inputs=inputs)
        break
    except Exception as exc:
        error_text = str(exc)
        is_temporary_gemini_error = any(code in error_text for code in ['429', '503', 'RESOURCE_EXHAUSTED', 'UNAVAILABLE'])

        if attempt == len(retry_delays) or not is_temporary_gemini_error:
            raise

        wait_seconds = retry_delays[attempt]
        print(f'Groq is temporarily unavailable or rate-limited. Retrying in {wait_seconds} seconds...')
        await asyncio.sleep(wait_seconds)



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6b43a3e4-1a84-4df9-864f-45a46762adc3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find potential leads in the Healthcare sector in United States. Use the Lead Database Lookup tool to     │
│  identify matching companies. Research each identified lead using at most one concise web search per company.   │
│  Collect available information such as company size, annual budget, industry fit, and business needs. Prepare   │
│  a short personalized outreach message for each lead. Keep findings concise and do not invent missing           │
│  information.                                                                                                   │
│  ID: 991dfb1e-154b-4107-b75e-e4812d7a553a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Query analysis failed, using defaults (complexity=simple): 1 validation error for QueryAnalysis
  Input should be a valid dictionary or instance of QueryAnalysis [type=model_type, input_value='{"keywords":["Healthcare...e"],"time_filter":null}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
C:\Users\jc\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:31<00:00, 2.60MiB/s]


╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 34296.94ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `qwen/qwen3.8-27b` in organization `org_01m0ykte2xe17b3d2rsbzs4r6j` service tier `on_demand` on input tokens per minute (ITPM): Limit 7000, Used 5738, Requested 1540. Please try again in 2.382857142s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `qwen/qwen3.8-27b` in organization `org_01m0ykte2xe17b3d2rsbzs4r6j` service tier `on_demand` on input tokens per minute (ITPM): Limit 7000, Used 5738, Requested 1540. Please try again in 2.382857142s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

Memory save analysis failed

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for     │
│  model `qwen/qwen3.8-27b` in organization `org_01m0ykte2xe17b3d2rsbzs4r6j` service tier `on_demand` on input    │
│  tokens per minute (ITPM): Limit 7000, Used 5738, Requested 1540. Please try again in 2.382857142s. Need more   │
│  tokens? Upgrade to Dev Tier today at                                                                           │
│  https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 170.40ms                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6b43a3e4-1a84-4df9-864f-45a46762adc3                                                                       │
│  Final Output: # Lead Qualification Report: Healthcare Sector (United States)                                   │
│                                                                                                                 │
│  **Date:** October 26, 2023                                                                                     │
│  **Prepared By:** Sales Intelligence Team                                                                       │
│  **Scope:** Identification, Research, Outreach, and Qualification of Healthcare Leads in the US                 │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  This report details the qualification of three identified leads in the United States Healthcare sector. Based  │
│  on research findings and a weighted scoring model (Budget, Need, Company Fit, Timeline), **MedTech             │
│  Solutions** and **BioCare Innovation** have been classified as **High Priority** leads, while **HealthPulse    │
│  AI** is classified as **Medium Priority**.                                                                     │
│                                                                                                                 │
│  *   **Top Opportunity:** MedTech Solutions (Score: 73/100) – High alignment between AI triage needs and        │
│  existing EHR/managed services model.                                                                           │
│  *   **High Potential:** BioCare Innovation (Score: 85/100) – Large budget and clear strategic expansion into   │
│  AI tools.                                                                                                      │
│  *   **Niche Fit:** HealthPulse AI (Score: 62/100) – Strong specific need for diagnostic software, but          │
│  constrained by lower budget.                                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Lead Profiles & Research Findings                                                                        │
│                                                                                                                 │
│  ### Lead 1: MedTech Solutions                                                                                  │
│  *   **Industry:** Healthcare IT                                                                                │
│  *   **Location:** United States                                                                                │
│  *   **Size:** 250 Employees                                                                                    │
│  *   **Annual Budget:** $200,000                                                                                │
│  *   **Business Needs:** AI patient triage and healthc

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
print('=' * 60)
print('FINAL LEAD QUALIFICATION REPORT')
print('=' * 60)
print(result)

FINAL LEAD QUALIFICATION REPORT
# Lead Qualification Report: Healthcare Sector (United States)

**Date:** October 26, 2023
**Prepared By:** Sales Intelligence Team
**Scope:** Identification, Research, Outreach, and Qualification of Healthcare Leads in the US

## Executive Summary

This report details the qualification of three identified leads in the United States Healthcare sector. Based on research findings and a weighted scoring model (Budget, Need, Company Fit, Timeline), **MedTech Solutions** and **BioCare Innovation** have been classified as **High Priority** leads, while **HealthPulse AI** is classified as **Medium Priority**.

*   **Top Opportunity:** MedTech Solutions (Score: 73/100) – High alignment between AI triage needs and existing EHR/managed services model.
*   **High Potential:** BioCare Innovation (Score: 85/100) – Large budget and clear strategic expansion into AI tools.
*   **Niche Fit:** HealthPulse AI (Score: 62/100) – Strong specific need for diagnostic software,